# Pretrained DINOv2-S/14 and Fine-tuned DINOv2

Both independent experiments use `timm`'s official `vit_small_patch14_dinov2.lvd142m` model with pretrained DINOv2 weights. The first run adapts all pretrained parameters for 20 epochs. The second run starts again from the original pretrained weights, freezes the complete DINOv2 body, and trains only a fresh two-class classification head for 20 epochs. The shared `data/common_split_manifest.csv` provides 176 training and 45 validation images.

> Graph names: **DINOv2** for the pretrained full-model run and **Fine-tuned DINOv2** for the classifier-only run. The latter keeps every body tensor frozen.

In [ ]:
import csv
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

project_candidates = [
    Path.home() / 'Documents' / 'Paper replication',
    Path.home() / 'Desktop' / 'Paper replication',
]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / 'data' / 'common_split_manifest.csv').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the Paper replication project in Documents or Desktop.')
SHARED = PROJECT_ROOT / 'model_reproductions' / 'shared'
MODEL_FOLDER = PROJECT_ROOT / 'model_reproductions' / '03_dinov2_pretrained'
LOCAL_DEPS = MODEL_FOLDER / '.deps'
sys.path.insert(0, str(LOCAL_DEPS))
sys.path.insert(0, str(SHARED))

import timm
from training import set_seed, train_experiment, train_frozen_with_feature_cache

MODEL_NAME = 'vit_small_patch14_dinov2.lvd142m'

def build_pretrained_dinov2():
    set_seed(42)
    try:
        return timm.create_model(
            MODEL_NAME, pretrained=True, num_classes=2, img_size=224
        )
    except RuntimeError as error:
        raise RuntimeError(
            f'{MODEL_NAME} is unavailable in timm {timm.__version__}. '
            'Install the notebook-local timm dependencies using the command in the hand-off instructions.'
        ) from error

In [ ]:
model = build_pretrained_dinov2()
total = sum(parameter.numel() for parameter in model.parameters())
trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
assert total == trainable
print(f'Pretrained DINOv2 parameters: {total:,}; all {trainable:,} are trainable')

In [ ]:
train_experiment(
    model=model,
    phase='pretrained_all_parameters',
    last_stage=model.blocks[-1],
    output_dir=MODEL_FOLDER / 'results_dinov2',
    epochs=20,
    learning_rate=1e-5,
    batch_size=8,
    center_crop=False,
)

## Fine-tuned DINOv2 — classifier only

This independent run starts again from the original pretrained DINOv2 weights. The ViT-S/14 body is completely frozen and its features are extracted once. Only the new two-class classification head is trained for 20 independently numbered epochs.

In [ ]:
fine_tuned_model = build_pretrained_dinov2()

train_frozen_with_feature_cache(
    model=fine_tuned_model,
    output_dir=MODEL_FOLDER / 'results_fine_tuned',
    epochs=20,
    learning_rate=1e-3,
    image_batch_size=8,
    classifier_batch_size=16,
    center_crop=False,
)

## Independent 20-epoch accuracy comparison

The DINOv2 and Fine-tuned DINOv2 histories are shown side by side, each with its own epoch numbering from 1 to 20. Maximum training and validation accuracies are labelled directly on each panel.

In [ ]:
def read_accuracy_history(path):
    with path.open(newline='', encoding='utf-8') as file:
        rows = list(csv.DictReader(file))
    return (
        [float(row['train_accuracy']) for row in rows],
        [float(row['validation_accuracy']) for row in rows],
    )

dinov2_train, dinov2_validation = read_accuracy_history(
    MODEL_FOLDER / 'results_dinov2' / 'training_history.csv'
)
fine_tuned_train, fine_tuned_validation = read_accuracy_history(
    MODEL_FOLDER / 'results_fine_tuned' / 'training_history.csv'
)

train_color = '#1f77b4'
validation_color = '#ff7f0e'
figure, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

def plot_accuracy_phase(axis, train_values, validation_values, title):
    epochs = list(range(1, len(train_values) + 1))
    axis.plot(epochs, train_values, color=train_color, linewidth=2, label='Training')
    axis.plot(epochs, validation_values, color=validation_color, linewidth=2, label='Validation')
    for label, values, color, text_offset in (
        ('Training', train_values, train_color, -40),
        ('Validation', validation_values, validation_color, 14),
    ):
        index = int(np.argmax(values))
        epoch = epochs[index]
        value = values[index]
        axis.scatter(epoch, value, color=color, s=70, zorder=5, edgecolor='white')
        axis.annotate(
            f'Max {label}: {value:.1%}\nEpoch {epoch}',
            xy=(epoch, value), xytext=(0, text_offset), textcoords='offset points',
            ha='center', color=color, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=color),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor=color, alpha=0.9),
        )
    axis.set(title=title, xlabel='Epoch', ylim=(0, 1.08), xlim=(1, len(epochs)))
    axis.grid(alpha=0.25)
    axis.legend(loc='lower right')

plot_accuracy_phase(
    axes[0], dinov2_train, dinov2_validation,
    'DINOv2 (pretrained, 20 epochs)',
)
plot_accuracy_phase(
    axes[1], fine_tuned_train, fine_tuned_validation,
    'Fine-tuned DINOv2 — classifier only (20 epochs)',
)
axes[0].set_ylabel('Accuracy')
dinov2_best = max(dinov2_validation)
fine_tuned_best = max(fine_tuned_validation)
figure.suptitle(
    f'Best observed validation accuracy: {dinov2_best:.1%} → {fine_tuned_best:.1%} '
    f'({(fine_tuned_best - dinov2_best) * 100:+.1f} percentage points)'
)
figure.tight_layout()
comparison_path = MODEL_FOLDER / 'dinov2_accuracy_comparison.png'
figure.savefig(comparison_path, dpi=220, bbox_inches='tight')
print('Saved:', comparison_path)

The saved scores are validation accuracies, not independent test accuracies. The same 45 validation images are used for checkpoint selection in both phases, so the best observed fine-tuned result may be optimistic.